# Notebook 2b -- Flows for lattice field theory: φ⁴ (~45 min)

This is the physics track.  We train a **CNF** (continuous normalizing
flow -- an ODE $dx/dt = v_\theta(x, t)$ with a neural velocity field) to
sample a 2D lattice φ⁴ theory by reverse KL, without any pre-existing dataset.
We then use the flow's exact density to turn it into a bias-free sampler
through reweighting, and meet one primary failure mode (mode-missing)
and how architecture can fix it in this setting.

We build on the transport framing of Notebook 1, so work through it first (at least
the top).  This is also the where we will reuse some of the tools in bijx,
rather than reimplementing everything here.

Three markers:
- ✏️ marks an exercise
- 📦 marks provided code or context (just read/run)
- ⭐ marks optional extra material

## 0. Lattice QFT refresher (📦)

1. Prediction in lattice field theory derive from an expectation
   $\langle O \rangle$ under $p(\phi) \propto e^{-S(\phi)}$, a Boltzmann
   distribution whose energy function $S$ we can write down explicitly.
   From our point of view, the whole subject is a *sampling problem*.
2. A (discretized) field configuration $\phi$ is just an $L \times L$ array of floats.
3. There is no dataset.  Unlike every model so far, however, the
   unnormalized density is known in closed form.

Three terms we will use throughout:

- **MCMC** (Markov chain Monte Carlo) samples by taking a local random
  walk whose stationary distribution is the target; it is a standard tool in
  lattice field theory (and many other sampling problems).
- **Reverse KL** is $D_{\mathrm{KL}}(\text{model} \,\|\, \text{target})$,
  with $D_{\mathrm{KL}}(p_1 \| p_2) = \int p_1 \log(p_1/p_2)$.
  It is *mode-seeking*: the model is penalized where it puts mass and the
  target has none, but not for missing target mass entirely.  (Forward
  KL, equivalent to maximum likelihood, is mode-covering.)
- **ESS** (effective sample size) measures how "statistically valuable" the samples
  are. If we draw exactly from the target it becomes 1 (or 100%), if we only repeat
  one sample for example (completely collapsed model), it would approach 0.

In [ ]:
# 📦 Canonical preamble + repo paths.
import time
from functools import partial
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import flax.nnx as nnx
import optax
import bijx
import matplotlib.pyplot as plt
from tqdm import tqdm

import iaifi_gm as gm  # helper package

rngs = nnx.Rngs(0)
gm.plotting.use_style()
gm.plotting.device_report()

# Repo-relative paths (works from the repo root or from notebooks/).
REPO = Path.cwd() if (Path.cwd() / "checkpoints").exists() else Path.cwd().parent
CKPT_DIR = REPO / "checkpoints"
for name in ["phi4_equivariant.msgpack", "phi4_broken.msgpack", "phi4_hmc_reference.npz"]:
    assert (CKPT_DIR / name).exists(), f"missing shipped artifact {CKPT_DIR / name}"
print(f"checkpoint dir: {CKPT_DIR} -- all φ⁴ artifacts found")

## 1. The physics target

Here is the theory, in one line:

$$ S(\phi) = \sum_x \Big[ \sum_{\mu=1}^{2} (\phi_{x+\hat\mu} - \phi_x)^2
   + m^2 \phi_x^2 + \lambda \phi_x^4 \Big], \qquad
   p(\phi) \propto e^{-S(\phi)}, $$

on a periodic $L \times L$ lattice ($x$ runs over sites, $\hat\mu$ over the
two lattice directions).  With $m^2 < 0$ each site sits in a double-well
potential $m^2\phi^2 + \lambda\phi^4$, and the action has an exact global
$\mathbb Z_2$ symmetry $\phi \to -\phi$.  In the broken phase the neighbor
coupling aligns the sites, and $p(\phi)$ concentrates on two symmetric
modes: mostly-positive and mostly-negative configurations.

The observables we track here are the magnetization per site and its moments,

$$ \bar m = L^{-2} \sum_x \phi_x, \qquad
   \langle |\bar m| \rangle, \qquad
   U_4 = 1 - \frac{\langle \bar m^4 \rangle}{3 \langle \bar m^2 \rangle^2}
   \ \text{(Binder cumulant)} . $$

**Why generative models?**  MCMC explores this distribution by local
steps, so successive samples are correlated, and near criticality (and
toward the continuum limit) the autocorrelation time rises rapidly
("critical slowing down").  A trained flow instead produces *independent*
samples in one shot, **with a tractable exact density** $q_\theta$.
In principle, it thus gives a path to overcome critical sampling.
How to scale it efficiently to large systems of interest is still an open problem.

In [ ]:
# 📦 Couplings, fixed for the rest of the notebook, and the action.
L = gm.phi4.L_DEFAULT           # 6  -> 36-dimensional target
M2, LAM = gm.phi4.M2, gm.phi4.LAM   # -1.0, 0.5 (broken phase)
print(f"L = {L}, m^2 = {M2}, λ = {LAM}")

phi_random = jax.random.normal(rngs(), (4, L, L))
print("S(φ) on 4 random fields:", np.round(np.asarray(gm.phi4.action(phi_random)), 2))

# MCMC reference numbers (with error bars) at these couplings,
# generated by scripts/hmc_phi4_reference.py
reference = gm.phi4.REFERENCE[(L, M2, LAM)]
for key, (val, err) in ((k, v) for k, v in reference.items() if k != "n_samples"):
    print(f"  reference {key:>9}: {val:.5f} ± {err:.5f}")

In [ ]:
# 📦 The two faces of the target: the per-site double well, and the bimodal
# magnetization histogram of the reference sampler.
hmc_ref = np.load(CKPT_DIR / "phi4_hmc_reference.npz")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
phi_grid = np.linspace(-1.8, 1.8, 200)
axes[0].plot(phi_grid, M2 * phi_grid**2 + LAM * phi_grid**4)
axes[0].set_xlabel(r"$\phi_x$")
axes[0].set_ylabel(r"$m^2 \phi_x^2 + \lambda \phi_x^4$")
axes[0].set_title("per-site double well")
axes[1].stairs(hmc_ref["density"], hmc_ref["bin_edges"], fill=True, alpha=0.6)
axes[1].set_xlabel(r"magnetization per site $\bar m$")
axes[1].set_ylabel("density")
axes[1].set_title(f"MCMC reference, {int(hmc_ref['n_samples']):,} samples")
fig.tight_layout()

The histogram should show two well-separated modes at $\bar m \approx \pm 0.85$,
related by the exact Z_2 symmetry; the density between them is a few
percent of the peak height.

## 2. Reverse-KL training -- no data needed

Everything so far in this tutorial minimized a regression loss over a
*dataset*.  Here there is none (although for small targets here we could "cheat"
and run standard MCMC): the target is known only as the unnormalized density $e^{-S}$.
So we train on the model's **own samples**:

$$ L(\theta)
   = \mathbb{E}_{\phi \sim q_\theta}\big[ \log q_\theta(\phi) + S(\phi) \big]
   = D_{\mathrm{KL}}(q_\theta \,\|\, p) - \log Z \;\ge\; -\log Z . $$

This is the **free-energy bound**: the loss estimates the free energy
$-\log Z$ from above, with equality iff $q_\theta = p$.

**The model** (📦) has three ingredients; this exact configuration trains
in ~300 steps.

- `bijx.ConvVF` is a convolutional CNF velocity field on the lattice.
  It is translation-equivariant by construction, and its default
  features (odd sine features, linear polynomial features, no bias) make
  it **Z_2-equivariant** too: $v_\theta(-\phi, t) = -v_\theta(\phi, t)$.
  Crucially, its divergence comes out **analytically**, at the cost of
  ~one forward pass; the ⭐ cell at the end of this section measures what that buys.
- The prior is white noise pushed through `bijx.FreeTheoryScaling` -- the
  exact free-field ($\lambda = 0$, $m^2 = 1$) distribution.  It handles
  the UV smoothing (damping high-momentum modes), so the flow only has to
  build the double-well and higher order structure.
- `bijx.ContFlowRK4(vf, steps=16)` integrates samples *and* log-density
  with fixed-step Runge-Kutta, from $t = 0$ (prior) to $t = 1$ (target).
  In production you would use an adaptive solver (`bijx.ContFlowDiffrax`);
  a fixed-step solver might be "exploited" by the optimizer (see training-cell).

In [ ]:
# 📦 Model construction.
from bijx.nn.features import FourierFeatures, PolynomialFeatures

FREE_PRIOR_M2 = 1.0  # mass^2 of the free-theory prior
RK4_STEPS = 16


def build_phi4_flow(*, broken: bool, seed: int = 0) -> bijx.Transformed:
    """Flow model q_θ over (L, L) fields; `broken=True` adds Z2-breaking
    features (bias + even polynomial powers)"""
    if broken:
        vf = bijx.ConvVF.build(
            (5, 5), (),
            use_bias=True,
            features=(
                partial(FourierFeatures, 49),
                partial(PolynomialFeatures, (0, 1, 2)),
            ),
            rngs=nnx.Rngs(params=seed),
        )
    else:  # defaults are Z2-equivariant: v(-φ, t) = -v(φ, t)
        vf = bijx.ConvVF.build((5, 5), (), rngs=nnx.Rngs(params=seed))
    prior = bijx.Transformed(
        bijx.IndependentNormal((L, L), rngs=nnx.Rngs(sample=seed + 1)),
        bijx.FreeTheoryScaling(FREE_PRIOR_M2, (L, L), half=False),
    )
    return bijx.Transformed(prior, bijx.ContFlowRK4(vf, steps=RK4_STEPS))


model = build_phi4_flow(broken=False, seed=0)
phi, log_q = model.sample((3,))
print(f"sample: phi {phi.shape}, log_q {log_q.shape}  (untrained)")

### ✏️ Exercise 1 -- the reverse-KL loss

Implement the estimator:

$$ L(\theta) = \mathbb{E}_{\phi \sim q_\theta}
   \big[ \log q_\theta(\phi) + S(\phi) \big]
   \;\approx\; \frac{1}{B} \sum_{i=1}^{B}
   \big[ \log q_\theta(\phi_i) + S(\phi_i) \big],
   \qquad \phi_i \sim q_\theta . $$

**Gradients must flow through the flow-generated samples** -- that is the
"reparametrization trick" popularized by VAEs, and it is what makes
$\nabla_\theta \mathbb{E}_{\phi \sim q_\theta}[\cdot]$ cheap here.
Do **not** apply `stop_gradient` to anything.

The shapes are `phi: (B, L, L)`, `log_q: (B,)`, and `target_ld: (B,)`;
the loss is a scalar.

In [ ]:
def reverse_kl_loss(model, batch_size: int):
    """Reverse-KL loss (~ free energy) and batch ESS/N.

    model.sample((B,)) -> phi: (B, L, L), log_q: (B,).
    Define target_ld: (B,) = -S(phi), then the scalar loss.
    -> (loss: scalar, ess: scalar in (0, 1])
    """
    phi, log_q = model.sample((batch_size,))
    raise NotImplementedError  # YOUR CODE HERE
    # Batch ESS/N as a training monitor -- §3 makes you build this yourself.
    ess = bijx.effective_sample_size(target_ld, log_q)
    return loss, ess

In [ ]:
# 📦 Shape check -- run before training.
loss_t, ess_t = reverse_kl_loss(model, 8)
assert jnp.shape(loss_t) == (), f"loss must be a scalar, got shape {jnp.shape(loss_t)}"
assert bool(jnp.isfinite(loss_t)), "loss is not finite"
assert 0.0 < float(ess_t) <= 1.0, f"ESS/N must be in (0, 1], got {float(ess_t)}"
grads_t = nnx.grad(lambda m: reverse_kl_loss(m, 8)[0])(model)
gnorm = float(optax.global_norm(nnx.state(grads_t)))
assert gnorm > 0, "zero gradient -- did you stop_gradient the samples?"
print(f"untrained loss {float(loss_t):.1f}, batch ESS/N {float(ess_t):.3f}, |grad| {gnorm:.2f}")

📦 **This is the one live training of the notebook**: ~300 steps, about
3-5 minutes on a modern laptop.

- **Warmup-cosine learning rate, decayed fully to 0**: in a 300-step
  budget a flat small LR does not converge -- and a hot flat LR eventually
  learns to *exploit the fixed-step RK4 integration error*: the reported
  loss sinks below the free-energy bound while true sample quality
  collapses. That is the reason for either using more steps, or even better
  use an adaptive step size sovler.

If training takes too long, feel free to skip and use the checkpoint below.

In [ ]:
N_STEPS = 300
BATCH = 128
PEAK_LR = 3e-3

schedule = optax.warmup_cosine_decay_schedule(0.0, PEAK_LR, 20, N_STEPS)
tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(schedule))
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)


def train_step(model, optimizer):
    (loss, ess), grads = nnx.value_and_grad(reverse_kl_loss, has_aux=True)(model, BATCH)
    optimizer.update(grads=grads, model=model)
    return loss, ess


losses = np.full(N_STEPS, np.nan)
ess_hist = np.full(N_STEPS, np.nan)
for i in tqdm(range(N_STEPS), desc="reverse KL"):
    loss, ess = train_step(model, optimizer)
    losses[i], ess_hist[i] = loss, ess
print(f"loss {losses[0]:.1f} -> {losses[-20:].mean():.2f} "
      f"(free-energy estimate), batch ESS/N -> {ess_hist[-20:].mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(losses)
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss  (≥ −log Z)")
axes[1].plot(ess_hist)
axes[1].set_xlabel("step"); axes[1].set_ylabel("batch ESS/N")
axes[1].set_ylim(0, 1)
fig.tight_layout()

In [ ]:
# 📦 If you interrupted the training cell above,
# flip the flag to swap in the shipped checkpoint.
USE_SHIPPED_CHECKPOINT = False
if USE_SHIPPED_CHECKPOINT:
    model = gm.checkpoints.load(
        build_phi4_flow(broken=False, seed=0), CKPT_DIR / "phi4_equivariant.msgpack"
    )
    print("loaded shipped phi4_equivariant.msgpack into `model`")

In [ ]:
# 📦 A few flow-sampled configurations and the raw
# magnetization histogram against the MCMC reference.
N_EVAL = 8192
phi_eval, log_q_eval = model.sample((N_EVAL,))
target_ld_eval = -gm.phi4.action(phi_eval)
mag_eval = np.asarray(gm.phi4.magnetization(phi_eval))

fig, axes = plt.subplots(1, 7, figsize=(11, 2.1),
                         gridspec_kw={"width_ratios": [1] * 6 + [2.6]})
for k in range(6):
    axes[k].imshow(np.asarray(phi_eval[k]), cmap="RdBu_r", vmin=-1.6, vmax=1.6)
    axes[k].set_xticks([]); axes[k].set_yticks([])
    axes[k].set_title(f"$\\bar m$={mag_eval[k]:+.2f}", fontsize=8)
axes[6].stairs(hmc_ref["density"], hmc_ref["bin_edges"], fill=True, alpha=0.4,
               label="MCMC ref.")
axes[6].hist(mag_eval, bins=40, density=True, histtype="step", lw=1.5,
             label="flow (raw)")
axes[6].set_xlabel(r"$\bar m$"); axes[6].legend(fontsize=7)
fig.tight_layout()
print(f"raw flow samples: <m> = {mag_eval.mean():+.3f}, "
      f"frac(m > 0) = {(mag_eval > 0).mean():.2f}  (both modes covered?)")

Both modes are populated and roughly symmetric -- but the histogram does
not match the reference *exactly*.
 Raw flow samples are draws from $q_\theta \ne p$, and §3 fixes this.

### ⭐ Measure the divergence cost

The log-density update needs $\nabla \cdot v$ along the trajectory.
`ConvVF` produces it analytically alongside $v$.  The generic fallback,
`bijx.AutoJacVF`, differentiates the velocity field -- one pass per
dimension, $O(D)$.  Time one call of each on the same vector-field body:

In [ ]:
vf_body = model.bijection.vf  # the trained ConvVF inside ContFlowRK4
auto_vf = bijx.AutoJacVF(lambda t, x: vf_body(t, x)[0], event_dim=2)

conv_call = jax.jit(lambda t, x: vf_body(t, x))
auto_call = jax.jit(lambda t, x: auto_vf(t, x))
phi_bench = jax.random.normal(jax.random.key(0), (BATCH, L, L))
for f, tag in [(conv_call, "ConvVF (analytic divergence)"), (auto_call, "AutoJacVF (autodiff trace) ")]:
    v, d = jax.block_until_ready(f(0.5, phi_bench))  # compile once
    t0 = time.perf_counter()
    for _ in range(10):
        jax.block_until_ready(f(0.5, phi_bench))
    print(f"{tag}: {(time.perf_counter() - t0) / 10 * 1e3:6.1f} ms per VF call")
v1, d1 = conv_call(0.5, phi_bench)
v2, d2 = auto_call(0.5, phi_bench)
assert jnp.allclose(d1, d2, atol=1e-3), "the two divergences should agree!"
print(f"divergences agree -- here D = {L * L}; at real lattice sizes the gap is fatal")

## 3. Exactness: reweighting & ESS

Here is the flow's strength over a data-trained generative model: for
every sample we know both $\log q_\theta(\phi)$ and
$\log p(\phi) + \log Z = -S(\phi)$ exactly.
We attach an **importance weight** to each sample,

$$ \log w_i = -S(\phi_i) - \log q_\theta(\phi_i), $$

and any expectation becomes exact in the limit of infinite samples:

$$ \langle O \rangle_p = \frac{\sum_i w_i\, O(\phi_i)}{\sum_i w_i}
   \quad (\text{self-normalized importance sampling}). $$

The price is variance: if $q_\theta$ matches $p$ poorly, a few huge
weights dominate.  The standard diagnostic is the effective sample size

$$ \mathrm{ESS} = \frac{\big(\sum_i w_i\big)^2}{\sum_i w_i^2},
   \qquad \frac{\mathrm{ESS}}{N} \in (0, 1], $$

the number of "equally-weighted" samples your $N$ weighted ones are "worth".

### ✏️ Exercise 2 -- ESS from log-weights

Implement the formula above given `log_w: (N,)`, returning the fraction
ESS/N.  One point of numerical stability: exponentiate *shifted* log-weights,
subtracting `jax.scipy.special.logsumexp(log_w)` (or `log_w.max()`)
first.  The ratio is invariant under any constant shift, so the unknown
$\log Z$ never matters.

In [ ]:
def ess_fraction(log_w):
    """ESS/N from log-weights.  log_w: (N,) -> scalar in (0, 1]."""
    raise NotImplementedError  # YOUR CODE HERE

In [ ]:
# 📦 Correctness checks.
n = 1000
assert jnp.isclose(ess_fraction(jnp.zeros(n)), 1.0), "uniform weights must give ESS/N = 1"
one_hot = jnp.where(jnp.arange(n) == 0, 0.0, -1e9)
assert jnp.isclose(ess_fraction(one_hot), 1.0 / n), "a single dominant weight must give ESS/N = 1/N"
assert jnp.isclose(  # shift invariance = log Z independence
    ess_fraction(jnp.arange(5.0)), ess_fraction(jnp.arange(5.0) + 123.0)
), "ESS must be invariant under constant log-weight shifts"
print("ess_fraction looks right")

📦 Below is the reweighted estimator. We apply it to the §2 samples and
compare against the shipped high-precision MCMC reference, which
`scripts/hmc_phi4_reference.py` generated from 64
independent chains with cross-chain error bars.

In [ ]:
def reweighted_mean(obs, log_w):
    """Self-normalized importance-sampling estimate of <obs>_p.

    obs: (N,), log_w: (N,) -> scalar.
    """
    log_w = log_w - jax.scipy.special.logsumexp(log_w)
    return jnp.sum(jnp.exp(log_w) * obs)


def observable_report(phi, log_q, label):
    """Print raw vs reweighted <|m|>, <m^2>, U4 (+ ESS) vs the MCMC reference."""
    target_ld = -gm.phi4.action(phi)
    log_w = target_ld - log_q
    mag = gm.phi4.magnetization(phi)
    ess = float(ess_fraction(log_w))
    m2_rw = reweighted_mean(mag**2, log_w)
    m4_rw = reweighted_mean(mag**4, log_w)
    rows = {
        "abs_mag": (float(jnp.mean(jnp.abs(mag))), float(reweighted_mean(jnp.abs(mag), log_w))),
        "mag_sq": (float(jnp.mean(mag**2)), float(m2_rw)),
        "binder_u4": (float(gm.phi4.binder_u4(mag)), float(1 - m4_rw / (3 * m2_rw**2))),
    }
    print(f"{label}  (N = {len(mag)}, ESS/N = {ess:.3f})")
    print(f"  {'observable':>10} {'raw':>9} {'reweighted':>11} {'reference':>19}")
    for key, (raw, rw) in rows.items():
        val, err = reference[key]
        print(f"  {key:>10} {raw:9.4f} {rw:11.4f} {val:12.5f} ± {err:.5f}")
    return log_w, mag


log_w_eval, _ = observable_report(
    phi_eval, log_q_eval, f"trained flow, fresh {N_EVAL}-sample batch"
)

The reweighted values should land on the reference within their statistical error
of ~0.03-0.04, set by the mere ~10^2 effective samples in this batch
(errors shrink as $1/\sqrt{\mathrm{ESS} \cdot N}$) -- where the *raw* means
are off by ~0.15, several times that.  **Reweighting makes the flow exact
for these observables**; more samples is the only thing between this and
the fifth digit.

Two caveats about ESS are worth recording.

- The training curve's batch-128 ESS is systematically *optimistic*;
  the weights are heavy-tailed, so small batches miss the tail.  Always
  quote ESS from a large fresh batch (as above), and expect it to
  fluctuate between batches.
- ESS measures *weight variance where the model has samples*.  It says
  nothing about regions with no samples at all.  That blind spot is important.

## 4. Mode-missing: the reverse-KL failure mode

Reverse KL is mode-seeking: a model covering just *one* of the two
$\pm\bar m$ modes pays no penalty for the missing one -- the expectation in
$L(\theta)$  runs over the model's own samples, which never visit it (in finite time).
The equivariant flow above cannot collapse by construction (a
Z_2-antisymmetric velocity field pushes a symmetric prior to an exactly
symmetric $q_\theta$).  So we *deliberately broke* the symmetry --
`use_bias=True` plus even polynomial features in the same `ConvVF` -- and
trained with a fixed seed in a noisy regime (small batch, hot LR, no
gradient clipping) until the flow tipped into a single mode.
Both checkpoints are in the
repo (`scripts/train_phi4_broken.py`, `scripts/train_phi4_equivariant.py`),
so this section just loads and inspects them.

**Diagnose with Z_2-odd observables.**  At finite volume the symmetry is
exact, so $\langle \bar m \rangle = 0$ identically.
Any statistically significant $\langle \bar m \rangle \ne 0$ (or a
lopsided $\mathrm{frac}(\bar m > 0)$) is a sign of mode collapse.

In [ ]:
# 📦 Load both shipped checkpoints
flow_broken = gm.checkpoints.load(
    build_phi4_flow(broken=True, seed=0), CKPT_DIR / "phi4_broken.msgpack"
)
flow_equiv = gm.checkpoints.load(
    build_phi4_flow(broken=False, seed=0), CKPT_DIR / "phi4_equivariant.msgpack"
)

diagnostics = {}
for label, flow in [("broken-Z2", flow_broken), ("equivariant", flow_equiv)]:
    phi_c, log_q_c = flow.sample((N_EVAL,))
    log_w_c = -gm.phi4.action(phi_c) - log_q_c
    mag_c = gm.phi4.magnetization(phi_c)
    diagnostics[label] = dict(
        mag=np.asarray(mag_c),
        log_w=np.asarray(log_w_c - jax.scipy.special.logsumexp(log_w_c)),
        ess=float(ess_fraction(log_w_c)),
        mean_m=float(reweighted_mean(mag_c, log_w_c)),
        frac_pos=float((mag_c > 0).mean()),
        abs_m=float(reweighted_mean(jnp.abs(mag_c), log_w_c)),
        u4=None,
    )
    m2_rw = reweighted_mean(mag_c**2, log_w_c)
    diagnostics[label]["u4"] = float(1 - reweighted_mean(mag_c**4, log_w_c) / (3 * m2_rw**2))
    # Rough 1σ error on <|m|>: weighted variance over the ESS·N effective samples.
    var_abs = float(m2_rw) - diagnostics[label]["abs_m"] ** 2
    diagnostics[label]["err_abs_m"] = float(
        np.sqrt(max(var_abs, 0.0) / (diagnostics[label]["ess"] * N_EVAL))
    )

print(f"{'':>12} {'ESS/N':>6} {'frac(m>0)':>10} {'<m> rw':>8}   ← Z2-odd flags")
for label, d in diagnostics.items():
    print(f"{label:>12} {d['ess']:6.3f} {d['frac_pos']:10.2f} {d['mean_m']:+8.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (label, d) in zip(axes, diagnostics.items()):
    ax.stairs(hmc_ref["density"], hmc_ref["bin_edges"], fill=True, alpha=0.35,
              label="MCMC ref.")
    # reweighted histogram: bin the samples with their normalized weights
    ax.hist(d["mag"], bins=40, range=(-1.5, 1.5), weights=np.exp(d["log_w"]) / (3 / 40),
            histtype="step", lw=1.6, label="flow, reweighted")
    ax.set_title(f"{label}:  $\\langle\\bar m\\rangle$ = {d['mean_m']:+.2f}, "
                 f"frac($\\bar m$>0) = {d['frac_pos']:.2f}", fontsize=9)
    ax.set_xlabel(r"$\bar m$")
    ax.legend(fontsize=7)
fig.tight_layout()

The broken flow sits in a single mode -- $\mathrm{frac}(\bar m > 0) \approx
0.97$, reweighted $\langle \bar m \rangle$ far from 0 -- while the
equivariant flow shows the symmetric two-peak picture.

Note that the collapose only shows up in Z_2-odd observable.
For Z_2-even observables, a single mode is sufficient, and in particular we would
not be able to detect the collapse.

In [ ]:
# 📦 Z2-even observables of the *collapsed* flow vs reference -- all fine.
# ± is the rough 1σ statistical error from the ESS·N effective samples;
# the collapsed flow's tiny ESS makes its error bars wide, not wrong.
print(f"{'':>12} {'ESS/N':>6} {'<|m|> rw':>16} {'U4 rw':>7}   ← Z2-even: all look healthy")
for label, d in diagnostics.items():
    print(f"{label:>12} {d['ess']:6.3f} {d['abs_m']:8.3f} ± {d['err_abs_m']:.3f} {d['u4']:7.4f}")
print(f"{'reference':>12} {'--':>6} {reference['abs_mag'][0]:8.3f} ± {reference['abs_mag'][1]:.3f} "
      f"{reference['binder_u4'][0]:7.4f}")

**Symmetry-blind validation cannot see this failure.**  If your test suite
only checks even observables and ESS, a collapsed flow passes it.

One moral moral is that we should try to build symmetries belong in the architecture.
The fix was not more training or a better loss -- it was the equivariant vector field,
which makes collapse *impossible* rather than unlikely.  However, this may not
always be possible or easy, especially if the mode structure of th target
is not known a priori.

### ⭐ Break it yourself

Set `RUN_BREAK_IT = True` below to retrain the broken architecture live
with §2's loop (~3 min).  Collapse is *seed-dependent* -- optimization
noise decides which mode wins, if any: seed 0 collapses, while seeds 1
and 2 happen to find the symmetric solution anyway.  Try a few.

In [ ]:
RUN_BREAK_IT = False   # flip to True to retrain live (~3 min)
BREAK_SEED = 0         # 0 collapses; try 1, 2, ...

if not RUN_BREAK_IT:
    print("[skip] set RUN_BREAK_IT = True to retrain the broken flow live "
          f"(~3 min); the shipped checkpoint above was trained with seed {0}.")
else:
    model_b = build_phi4_flow(broken=True, seed=BREAK_SEED)
    # Noisy regime (this is what tips it over): batch 64, hot LR, no clipping.
    tx_b = optax.adam(optax.warmup_cosine_decay_schedule(0.0, 5e-3, 20, N_STEPS))
    opt_b = nnx.Optimizer(model_b, tx_b, wrt=nnx.Param)

    def train_step_b(model, optimizer):  # no @nnx.jit (see §2)
        (loss, ess), grads = nnx.value_and_grad(reverse_kl_loss, has_aux=True)(model, 64)
        optimizer.update(grads=grads, model=model)
        return loss, ess

    for i in tqdm(range(N_STEPS), desc=f"broken, seed {BREAK_SEED}"):
        train_step_b(model_b, opt_b)
    phi_b, log_q_b = model_b.sample((N_EVAL,))
    mag_b = np.asarray(gm.phi4.magnetization(phi_b))
    print(f"seed {BREAK_SEED}: <m> raw = {mag_b.mean():+.3f}, "
          f"frac(m > 0) = {(mag_b > 0).mean():.2f}  (collapsed if far from 0.5)")

## 5. Pointers (📦, take-home)

- **Flows for lattice QFT, the field:** discrete coupling-layer flows
  ([Albergo, Kanwar & Shanahan, 2019](https://arxiv.org/abs/1904.12072)),
  and gauge-equivariant versions
  ([Kanwar et al., 2020](https://arxiv.org/abs/2003.06413)).
- Near the continuum limit the sampling cost of MCMC
  blows up, through critical slowing down and topological freezing
  ([Schaefer et al., 2010](https://arxiv.org/abs/1009.5228)).  Flows trade
  that for an up-front training cost; whether that trade scales is an
  active research question ([Abbott et al., 2022](https://arxiv.org/abs/2211.07541)).
- **CNFs on lattices**, this notebook's route, combine continuous flows
  with analytic divergence and equivariance as an architecture property --
  see the [bijx documentation](https://mathisgerdes.github.io/bijx/)
  ([code](https://github.com/mathisgerdes/bijx)) for the building blocks
  used here, including the scalar-theory and SU(3) tutorials this
  notebook draws on.

## ⭐ Appendix: one SU(3) variable

Lattice *gauge* theories live on group manifolds: each link variable is a
unitary matrix $U \in SU(3)$.  This appendix (all provided code)
repeats the §2 recipe for a single SU(3) variable:

- **The target** is $p(U) \propto e^{\beta\, \mathrm{Re}\,\mathrm{tr}\, U}$,
  a one-link toy version of the Wilson gauge action, invariant under
  conjugation $U \to V U V^\dagger$.
- **The prior** is the **Haar measure**, the unique "uniform"
  distribution on the group (`bijx.lie.HaarDistribution`).
- **The flow**'s velocity field is the *gradient of a learned scalar
  potential* built from conjugation-invariant features
  ($\mathrm{Re}\,\mathrm{tr}\,U^k$ and $\mathrm{Im}\,\mathrm{tr}\,U^k$)
  so the flow is symmetry-equivariant by construction.
  `bijx.lie.value_grad_divergence` returns the gradient *and*
  divergence on the group (Lie-algebra directional derivatives along the
  anti-Hermitian generators `bijx.lie.SU3_GEN`), and a Crouch-Grossmann
  integrator (`bijx.ContFlowCG`) keeps every step exactly on SU(3).
- **The loss** is your Exercise-1 reverse KL.

In [ ]:
# 📦 Target + potential + vector field on the group.
from jax_autovmap import autovmap

BETA = 2.0


def su3_target_log_density(u):
    """log p(U) = β Re tr U (unnormalized); conjugation-invariant."""
    return BETA * jnp.trace(u, axis1=-2, axis2=-1).real


class Potential(nnx.Module):
    """Scalar potential on conjugation-invariant features Re/Im tr U^k."""

    def __init__(self, width=64, *, rngs):
        self.lin1 = nnx.Linear(5, width, rngs=rngs)
        self.lin2 = nnx.Linear(width, width, rngs=rngs)
        self.out = nnx.Linear(
            width, 1, kernel_init=nnx.initializers.normal(1e-3), rngs=rngs
        )

    def __call__(self, t, u):
        tr1 = jnp.trace(u, axis1=-2, axis2=-1)
        tr2 = jnp.trace(u @ u, axis1=-2, axis2=-1)
        feats = jnp.stack(
            [tr1.real, tr1.imag, tr2.real, tr2.imag, jnp.asarray(t, tr1.real.dtype)]
        )
        h = nnx.gelu(self.lin1(feats))
        h = h + nnx.gelu(self.lin2(h))
        return self.out(h).squeeze()


class PotentialVF(nnx.Module):
    """CNF vector field on SU(3): (gradient, -divergence) of the potential."""

    def __init__(self, potential):
        self.potential = potential

    @autovmap(t=0, u=2)
    def __call__(self, t, u):
        _, vec, div = bijx.lie.value_grad_divergence(
            partial(self.potential, t), u, bijx.lie.SU3_GEN
        )
        return vec, -div

In [ ]:
# 📦 Model: Haar prior -> Crouch-Grossmann CNF; reverse-KL training.
su3_flow = bijx.ContFlowCG(
    PotentialVF(Potential(rngs=nnx.Rngs(params=0))),
    tableau=bijx.cg.CG2,
    steps=10,
    x_type=bijx.cg.Unitary(
        transport_adjoint=True,
        derivative=bijx.cg.UnitaryDeriv(project_step=True),
    ),
)
su3_model = bijx.Transformed(
    bijx.lie.HaarDistribution(3, rngs=nnx.Rngs(sample=1)), su3_flow
)

SU3_STEPS = 300
su3_optimizer = nnx.Optimizer(
    su3_model,
    optax.chain(optax.clip_by_global_norm(1.0), optax.adam(2e-3)),
    wrt=nnx.Param,
)


@nnx.jit  # ContFlowCG's adjoint composes fine with jit-of-grad
def su3_train_step(model, optimizer):
    def loss_fn(model):
        u, log_q = model.sample((64,))
        return jnp.mean(log_q - su3_target_log_density(u))  # same reverse KL as §2

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads=grads, model=model)
    return loss


su3_losses = np.full(SU3_STEPS, np.nan)
for i in tqdm(range(SU3_STEPS), desc="SU(3)"):
    su3_losses[i] = su3_train_step(su3_model, su3_optimizer)

u, log_q = su3_model.sample((512,))
su3_ess = float(bijx.effective_sample_size(su3_target_log_density(u), log_q))
print(f"loss {su3_losses[0]:.3f} -> {su3_losses[-10:].mean():.3f}, "
      f"eval ESS/N = {su3_ess:.2f}")

📦 **Final figure.**  A conjugation-invariant density depends only
on the eigenvalues $e^{i\theta_1}, e^{i\theta_2}, e^{i\theta_3}$ (with
$\theta_3 = -\theta_1 - \theta_2$), so the whole distribution lives on a
2D torus of eigenvalue angles.  We evaluate the target and the trained
flow on that grid -- including the Haar volume factor (a Vandermonde
determinant), which is what makes "uniform on the group" non-uniform in
angles.

In [ ]:
GRID = 40

def su3_angle_density(density_fn):
    """(θ1, θ2)-grid density (Haar-weighted, normalized): -> (GRID, GRID)."""
    angles, dens, haar_w = bijx.lie.evaluate_density_on_eigenvalue_grid(
        jax.jit(density_fn), n=3, grid_points=GRID
    )
    d = np.asarray(dens * haar_w).reshape(GRID, GRID)
    return d / d.sum()


d_target = su3_angle_density(lambda u: jnp.exp(su3_target_log_density(u)))
d_flow = su3_angle_density(lambda u: jnp.exp(su3_model.log_density(u)))
d_haar = su3_angle_density(lambda u: jnp.ones(u.shape[:-2]))

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
vmax = max(d_target.max(), d_flow.max())
for ax, d, title, vm in [
    (axes[0], d_haar, "Haar prior", d_haar.max()),
    (axes[1], d_target, "target  $\\propto e^{\\beta \\mathrm{Re\\,tr}\\,U}$", vmax),
    (axes[2], d_flow, "trained flow", vmax),
]:
    im = ax.imshow(d.T, origin="lower", extent=(-np.pi, np.pi, -np.pi, np.pi),
                   vmin=0, vmax=vm, cmap="viridis")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(r"$\theta_1$")
axes[0].set_ylabel(r"$\theta_2$")
fig.tight_layout()

# Total-variation distance between the two normalized grid densities:
# 0 = identical, 1 = disjoint.
tv = 0.5 * float(np.abs(d_flow - d_target).sum())
print(f"total-variation distance flow ↔ target on the grid: {tv:.3f} "
      f"(~ {100 * tv:.0f}% of the mass misplaced after {SU3_STEPS} quick steps "
      "-- train longer for a sharper match)")

The trained flow matches the target density on the eigenvalue torus to
~10% in total variation after this quick training run (train longer for a
sharper match) -- the same reverse-KL transport recipe, on a curved
space with none of the $\mathbb{R}^n$ structure we started from.  From
here to lattice gauge theory is "only" a matter of many links and gauge
(rather than global conjugation) symmetry.